In [ ]:
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/sleep_stage_classification"
TRAIN_PATH = BASE + "/train/train"
TEST_PATH  = BASE + "/test_segment/test_segment"

In [ ]:
WINDOW = 200
STEP = 50

In [ ]:
def create_windows(df, window_size=200, step=50):
    X = []
    y = []

    for i in range(0, len(df) - window_size, step):
        window = df.iloc[i:i+window_size].copy()

        # 🔥 label (เก็บก่อน)
        label = window["Sleep_Stage"].iloc[len(window)//2]

        # 🔥 ลบ label ออกก่อนทำ feature
        window = window.drop(columns=["Sleep_Stage"])

        # 🔥 feature
        window["acc_mag"] = np.sqrt(
            window["ACC_X"]**2 + window["ACC_Y"]**2 + window["ACC_Z"]**2
        )

        feat = []
        feat += list(window.mean())
        feat += list(window.std())
        feat += list(window.max())
        feat += list(window.min())

        X.append(feat)
        y.append(label)

    return X, y

In [ ]:
import os
print(os.listdir(TRAIN_PATH))

['train001.csv', 'train002.csv', 'train003.csv', 'train004.csv', 'train005.csv', 'train006.csv', 'train008.csv', 'train007.csv', 'train009.csv', 'train010.csv', 'train012.csv', 'train011.csv', 'train014.csv', 'train013.csv', 'train016.csv', 'train015.csv', 'train018.csv', 'train017.csv', 'train020.csv', 'train019.csv', 'train021.csv', 'train022.csv', 'train023.csv', 'train024.csv', 'train026.csv', 'train025.csv', 'train027.csv', 'train028.csv', 'train030.csv', 'train029.csv', 'train031.csv', 'train032.csv', 'train034.csv', 'train033.csv', 'train036.csv', 'train035.csv', 'train038.csv', 'train037.csv', 'train040.csv', 'train039.csv', 'train041.csv', 'train042.csv', 'train044.csv', 'train043.csv', 'train046.csv', 'train045.csv', 'train048.csv', 'train047.csv', 'train049.csv', 'train050.csv', 'train052.csv', 'train051.csv', 'train054.csv', 'train053.csv', 'train056.csv', 'train055.csv', 'train057.csv', 'train058.csv', 'train060.csv', 'train059.csv', 'train061.csv', 'train062.csv', 'train0

In [ ]:
file_path = "/content/drive/MyDrive/sleep_stage_classification/train/train/train001.csv"

df = pd.read_csv(file_path)
print(len(df))

356640


In [ ]:
X_w, y_w = create_windows(df, WINDOW, STEP)
print(len(X_w))

7129


In [ ]:
X, y = [], []

for file in os.listdir(TRAIN_PATH):
    if not file.endswith(".csv"):
        continue

    df = pd.read_csv(os.path.join(TRAIN_PATH, file))

    X_w, y_w = create_windows(df, WINDOW, STEP)

    X.extend(X_w)
    y.extend(y_w)

In [ ]:
print(len(X), len(y))

640449 640449


In [ ]:
import numpy as np

X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

print(le.classes_)

['N1' 'N2' 'N3' 'R' 'W']


In [ ]:
print(len(y_enc))

640449


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200)
model.fit(X, y_enc)

RandomForestClassifier(n_estimators=200)

In [ ]:
import os
print(os.listdir(TEST_PATH))

['test005', 'test010', 'test008', 'test006', 'test009', 'test007', 'test004', 'test003', 'test001', 'test002']


In [ ]:
def extract_features(window):
    window = window.copy()

    window["acc_mag"] = np.sqrt(
        window["ACC_X"]**2 + window["ACC_Y"]**2 + window["ACC_Z"]**2
    )

    window = window.drop(columns=["Sleep_Stage"], errors="ignore")

    feat = []
    feat += list(window.mean())
    feat += list(window.std())
    feat += list(window.max())
    feat += list(window.min())

    return feat

In [ ]:
import os
import pandas as pd
import numpy as np
from collections import Counter

file_ids = []
final_preds = []

for folder in os.listdir(TEST_PATH):
    folder_path = os.path.join(TEST_PATH, folder)

    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)

        preds = []

        # 🔥 handle window
        if len(df) < WINDOW:
            feat = extract_features(df)
            final = model.predict([feat])[0]

        else:
            for i in range(0, len(df) - WINDOW, STEP):
                window = df.iloc[i:i+WINDOW]
                feat = extract_features(window)

                pred = model.predict([feat])[0]
                preds.append(pred)

            if len(preds) == 0:
                feat = extract_features(df)
                final = model.predict([feat])[0]
            else:
                final = Counter(preds).most_common(1)[0][0]

        file_ids.append(file.replace(".csv",""))
        final_preds.append(final)


# 🔥 แปลง label กลับ
pred_labels = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    "id": file_ids,
    "labels": pred_labels
})

submission.to_csv("submission.csv", index=False)

print("Total:", len(submission))
print(submission.head())

Total: 7832
              id labels
0  test005_00024     N2
1  test005_00025     N2
2  test005_00007      W
3  test005_00032      W
4  test005_00028      W


In [ ]:
pred_labels = le.inverse_transform(final_preds)

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "id": file_ids,
    "labels": pred_labels
})

submission.to_csv("submission.csv", index=False)

In [ ]:
print(len(submission))  # ต้อง ~7832
print(submission.head())
print(submission["labels"].value_counts())

0
Empty DataFrame
Columns: [id, labels]
Index: []
Series([], Name: count, dtype: int64)
